In [ ]:
import os
import sys
import subprocess
import time
import glob
import copy
import random
import tarfile
import logging
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from urllib.request import urlretrieve
from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

In [ ]:

# --- 1. LOGGING INFRASTRUCTURE ---
def setup_logger(name: str = "Factory_Segmentation"):
    logger = logging.getLogger(name)
    if logger.hasHandlers():
        logger.handlers.clear()
    logger.setLevel(logging.INFO)
    logger.propagate = False
    formatter = logging.Formatter('%(asctime)s - [%(levelname)s] - %(message)s', datefmt='%H:%M:%S')
    ch = logging.StreamHandler(sys.stdout)
    ch.setFormatter(formatter)
    logger.addHandler(ch)
    return logger

logger = setup_logger()

In [ ]:
# --- 2. CONFIGURATION ---
@dataclass
class SegmentationConfig:
    DATA_URL: str = "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420937370-1629958698/bottle.tar.xz"
    ARCHIVE_NAME: str = "bottle.tar.xz"
    DATA_ROOT: Path = Path("bottle")

    BATCH_SIZE: int = 16
    NUM_WORKERS: int = 2
    EPOCHS: int = 15
    LEARNING_RATE: float = 1e-4
    SEED: int = 42

    SAVE_PATH: str = "best_unet_bottle.pth"
    DEFECT_AREA_THRESHOLD: int = 50 # Minimum connected pixels to trigger an alarm

In [ ]:
# --- 3. CUSTOM LOSS & DATASET ---
class CompoundLoss(nn.Module):
    """Combines Cross Entropy for stability and Dice for spatial accuracy."""
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode='binary', from_logits=True)

    def forward(self, logits, targets):
        return 0.5 * self.bce(logits, targets) + 0.5 * self.dice(logits, targets)

class MVTecSegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        image = cv2.cvtColor(cv2.imread(self.image_paths[idx]), cv2.COLOR_BGR2RGB)
        mask_path = self.mask_paths[idx]

        if mask_path is not None:
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask = np.where(mask > 127, 1.0, 0.0).astype(np.float32)
        else:
            mask = np.zeros(image.shape[:2], dtype=np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image, mask = augmented['image'], augmented['mask']

        if mask.ndim == 2: mask = mask.unsqueeze(0)
        return image, mask.float()

In [ ]:
# --- 4. THE OOP CONTROLLER ---
class SemanticAOISystem:
    def __init__(self, config: SegmentationConfig):
        self.cfg = config
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self._set_seed()
        self.model = None
        logger.info(f"Computation Device: {self.device.type.upper()}")

    def _set_seed(self):
        random.seed(self.cfg.SEED)
        np.random.seed(self.cfg.SEED)
        torch.manual_seed(self.cfg.SEED)
        if torch.cuda.is_available():
            torch.backends.cudnn.deterministic = True

    def ingest_data(self):
        if not self.cfg.DATA_ROOT.exists():
            logger.info("Downloading Dataset...")
            urlretrieve(self.cfg.DATA_URL, self.cfg.ARCHIVE_NAME)
            with tarfile.open(self.cfg.ARCHIVE_NAME, "r:xz") as tar: tar.extractall()
            os.remove(self.cfg.ARCHIVE_NAME)

    def prepare_dataloaders(self):
        img_dir_test = self.cfg.DATA_ROOT / 'test'
        img_dir_train_good = self.cfg.DATA_ROOT / 'train' / 'good'

        defect_images, defect_masks = [], []
        for defect in ['broken_large', 'broken_small', 'contamination']:
            files = sorted(list((img_dir_test / defect).glob('*.png')))
            defect_images.extend([str(f) for f in files])
            defect_masks.extend([str(f).replace('test', 'ground_truth').replace('.png', '_mask.png') for f in files])

        train_def_img, val_def_img, train_def_mask, val_def_mask = train_test_split(
            defect_images, defect_masks, test_size=0.2, random_state=self.cfg.SEED
        )

        good_images = sorted([str(f) for f in img_dir_train_good.glob('*.png')])

        train_files, train_masks = train_def_img + good_images, train_def_mask + ([None] * len(good_images))
        val_files, val_masks = val_def_img, val_def_mask

        transforms = {
            'train': A.Compose([
                A.Resize(256, 256),
                A.HorizontalFlip(p=0.5),
                A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
                A.RandomBrightnessContrast(p=0.2),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
                ToTensorV2()
            ]),
            'val': A.Compose([
                A.Resize(256, 256), A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2()
            ])
        }

        self.dataloaders = {
            'train': DataLoader(MVTecSegmentationDataset(train_files, train_masks, transforms['train']), batch_size=self.cfg.BATCH_SIZE, shuffle=True, num_workers=self.cfg.NUM_WORKERS),
            'val': DataLoader(MVTecSegmentationDataset(val_files, val_masks, transforms['val']), batch_size=self.cfg.BATCH_SIZE, shuffle=False, num_workers=self.cfg.NUM_WORKERS)
        }
        self.dataset_sizes = {'train': len(train_files), 'val': len(val_files)}
        logger.info(f"Pipelines Ready. Train: {len(train_files)} | Val: {len(val_files)}")

    def build_model(self):
        self.model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=1, activation=None).to(self.device)
        self.criterion = CompoundLoss()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.cfg.LEARNING_RATE)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, mode='min', factor=0.1, patience=3)

    @staticmethod
    def _calculate_iou(pred_mask, true_mask, eps=1e-6):
        intersection = (pred_mask & true_mask).sum().float()
        union = (pred_mask | true_mask).sum().float()
        return (intersection + eps) / (union + eps)

    def train(self):
        if not self.model: self.build_model()
        best_wts = copy.deepcopy(self.model.state_dict())
        best_iou = 0.0

        logger.info(f"Starting Segmentation Training ({self.cfg.EPOCHS} Epochs)...")

        for epoch in range(self.cfg.EPOCHS):
            for phase in ['train', 'val']:
                self.model.train() if phase == 'train' else self.model.eval()
                running_loss, running_iou = 0.0, 0.0

                pbar = tqdm(self.dataloaders[phase], desc=f"Epoch {epoch+1} [{phase.upper()}]", leave=False)
                for inputs, masks in pbar:
                    inputs, masks = inputs.to(self.device), masks.to(self.device)
                    self.optimizer.zero_grad()

                    with torch.set_grad_enabled(phase == 'train'):
                        outputs = self.model(inputs)
                        loss = self.criterion(outputs, masks)

                        preds = (outputs > 0).byte()
                        true_masks = (masks > 0.5).byte()

                        if phase == 'train':
                            loss.backward()
                            self.optimizer.step()

                    batch_size = inputs.size(0)
                    running_loss += loss.item() * batch_size
                    running_iou += self._calculate_iou(preds, true_masks).item() * batch_size
                    pbar.set_postfix(loss=loss.item())

                epoch_loss = running_loss / self.dataset_sizes[phase]
                epoch_iou = running_iou / self.dataset_sizes[phase]

                if phase == 'val':
                    logger.info(f"Epoch {epoch+1}/{self.cfg.EPOCHS} | VAL Loss: {epoch_loss:.4f} | VAL IoU: {epoch_iou:.4f}")
                    self.scheduler.step(epoch_loss)

                    if epoch_iou > best_iou:
                        best_iou = epoch_iou
                        best_wts = copy.deepcopy(self.model.state_dict())
                        torch.save(self.model.state_dict(), self.cfg.SAVE_PATH)
                        logger.info(f"New Best Model Saved (IoU: {best_iou:.4f})")

        self.model.load_state_dict(best_wts)
        logger.info("Training Complete.")

    def load_model(self, path: str):
        if not self.model: self.build_model()
        self.model.load_state_dict(torch.load(path, map_location=self.device))
        logger.info(f"Weights loaded from {path}")

    def evaluate_image(self, image_path: str) -> dict:
        """Production API Inference Method."""
        if not self.model: raise RuntimeError("Model not loaded.")

        img = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
        transform = A.Compose([A.Resize(256, 256), A.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)), ToTensorV2()])

        input_tensor = transform(image=img)['image'].unsqueeze(0).to(self.device)

        self.model.eval()
        with torch.no_grad():
            logits = self.model(input_tensor)
            probs = torch.sigmoid(logits)
            pred_mask = (probs > 0.5).squeeze().cpu().numpy()

        defect_pixels = np.sum(pred_mask)
        is_defect = defect_pixels > self.cfg.DEFECT_AREA_THRESHOLD

        return {
            "is_defect": bool(is_defect),
            "defect_pixel_count": int(defect_pixels),
            "threshold": self.cfg.DEFECT_AREA_THRESHOLD
        }

    def visualize_predictions(self, num_images=4, save_path="segmentation_grid.png"):
        """QA Tool for overlaying masks."""
        if not self.model: return
        logger.info("Generating semantic overlays...")
        self.model.eval()

        fig, axs = plt.subplots(num_images // 2, 2, figsize=(10, 5 * (num_images // 2)))
        axs = axs.flatten()

        with torch.no_grad():
            inputs, masks = next(iter(self.dataloaders['val']))
            inputs, masks = inputs.to(self.device), masks.to(self.device)

            logits = self.model(inputs)
            preds = (torch.sigmoid(logits) > 0.5).float()

            for j in range(min(num_images, inputs.size(0))):
                img_np = np.clip(inputs[j].cpu().permute(1, 2, 0).numpy() * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]), 0, 1)
                pred_mask = preds[j].cpu().squeeze().numpy()
                true_mask = masks[j].cpu().squeeze().numpy()

                iou = self._calculate_iou(preds[j].byte(), masks[j].byte()).item()

                axs[j].imshow(img_np)
                axs[j].imshow(pred_mask, alpha=0.5, cmap='Reds') # Red = Prediction overlay
                axs[j].set_title(f"IoU: {iou:.2f}", fontweight='bold')
                axs[j].axis('off')

        plt.tight_layout()
        plt.savefig(save_path)
        logger.info(f"Overlay grid saved to: {save_path}")

In [ ]:
# --- 5. EXECUTION BLOCK ---
if __name__ == "__main__":
    config = SegmentationConfig(EPOCHS=35)
    system = SemanticAOISystem(config)

    system.ingest_data()
    system.prepare_dataloaders()
    system.train()

    prod_system = SemanticAOISystem(config)
    prod_system.load_model(config.SAVE_PATH)

    test_defect = list((config.DATA_ROOT / 'test' / 'broken_large').glob('*.png'))[0]
    result = prod_system.evaluate_image(str(test_defect))

    logger.info(f"API Smoke Test Payload: {result}")
    if result['is_defect']:
        logger.info("SYSTEM SECURED: Ready for Docker API.")
        prod_system.prepare_dataloaders()
        prod_system.visualize_predictions(num_images=4)